# 11. 目錄統計 I：完整度與規模分布

{doc}`第 10 章 <10_point_process>`把地震序列寫成了點過程：條件強度
$\lambda^*(t,x,y,m)$、對數概似 $\ln L=\sum\ln\lambda^*-\int\lambda^*$、
隨機時間變換與殘差診斷。那一章最後示範了一個最簡單的最大概似：
當規模服從指數密度 $s(m)=\beta e^{-\beta(m-m_0)}$ 時，$\hat\beta$
有封閉解。式子只有一行，看起來像整套機器裡最不需要擔心的零件。

這一章要說的是：那一行是整套機器裡**最容易出錯**的零件。它的兩個
前提——「規模是連續的」與「目錄是完整的」——在真實目錄裡都不成立，
而且各自埋了一個方向相反的偏差。而 $\beta$（等價地 $b$ 值）並不
孤立：它是 ETAS 條件強度的規模分量（第 13、14 章）、是 PSHA 的
規模–頻率輸入（第 21 章）、是「前震 b 值比較低」這類前兆宣稱的
主角。地基歪一度，上層建築就斜一片。

所以這章從最不浪漫的地方開始：**地震目錄的規模欄位不是一個物理量，
而是一段行政史**。接著把 GR 律、$M_c$ 三法、Aki 最大概似及其現代
修正版一路推導完。本章會用到第 10 章的對數概似框架與 Fisher 資訊，
其餘從頭建立；時間軸上的叢集律留給第 12 章。

In [ ]:
from gdms_toolkit.viz import setup_plotly
setup_plotly()

## 11.1 目錄是一段行政史

台灣的儀器目錄跨越半個世紀，這半世紀裡「規模」這一欄的定義換過
三次、測站數翻過兩番、資料擷取方式改過一次。跨越十年以上的統計
若沒有先處理這些換代，看到的「地震活動變化」有很大機率只是儀器
變化。整理觀測史的文獻（Wang et al. 2015；Chang et al. 2016；
Tsai et al. 2012）後，有**五個該切一刀的年份**：

| 年份 | 發生了什麼 | 對統計的影響 |
|---|---|---|
| 1973 | TTSN 啟用，用延時規模 $M_D$ | 儀器目錄起點，$M_c\approx2.6$ |
| 1987.6 | 延時規模由類比轉數位 | 造成 1985–1991 的 $M\ge5$ 空隙 |
| 1991.3 | 改用模擬 Wood–Anderson $M_L$ | 規模尺度換代 |
| 1994 | 觸發式改連續記錄 | 年偵測數 4,000 → 20,000 |
| 2012 | 觀測網升級成熟 | 年偵測數再翻倍到約 40,000 |

把長期目錄按這些切點分段、各畫一條規模–頻率分布（frequency–
magnitude distribution, FMD），偵測能力的躍進會直接顯示在曲線
往下彎的位置上：

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from gdms_toolkit import load_taiwan_catalog
from gdms_toolkit.download import CACHE_DIR
from gdms_toolkit.viz import ACCENT, PALETTE, QUAKE_COLOR, apply_layout

DM = 0.1            # 台灣 CWA 目錄的規模格距 ΔM
DELTA = DM / 2      # 半格 δ
L10 = np.log(10.0)

cat_long = load_taiwan_catalog()
cat_long = cat_long.assign(M=np.round(cat_long.ML.to_numpy() / DM) * DM)

ERAS = {"1973–1987.5（類比 M_D）": ("1973", "1987-06-01"),
        "1987.6–1991.2（數位 M_D）": ("1987-06-01", "1991-03-01"),
        "1994–2011（連續記錄 M_L）": ("1994", "2012"),
        "2012–2025（觀測網成熟）": ("2012", "2026")}
BINS = np.arange(1.95, 7.05, DM)
CENTERS = np.round(BINS[:-1] + DELTA, 2)


def maxc(m, corr=0.2):
    """最大曲率法（MAXC）＋慣用的 0.2 保守修正。"""
    counts, _ = np.histogram(np.asarray(m, float), bins=BINS)
    return round(float(CENTERS[np.argmax(counts)]) + corr, 2)


mags = np.arange(2.0, 7.01, DM)
fig = go.Figure()
era_mc = {}
for (label, (t0, t1)), color in zip(ERAS.items(), PALETTE):
    sub = cat_long[(cat_long.time >= t0) & (cat_long.time < t1)]
    yrs = (sub.time.max() - sub.time.min()).days / 365.25
    rate = np.array([(sub.M >= m - 1e-9).sum() / yrs for m in mags])
    era_mc[label] = maxc(sub.M.to_numpy())
    fig.add_trace(go.Scatter(x=mags, y=rate, mode="lines", name=label,
                             line=dict(color=color, width=2)))
    j = int(np.argmin(np.abs(mags - era_mc[label])))
    fig.add_trace(go.Scatter(x=[mags[j]], y=[rate[j]], mode="markers",
                             showlegend=False,
                             marker=dict(color=color, size=11,
                                         symbol="diamond",
                                         line=dict(color="white", width=1))))
apply_layout(fig,
             title="分年代累積 FMD 與各年代 Mc（菱形＝MAXC＋0.2："
                   + "、".join(f"{v:.1f}" for v in era_mc.values()) + "）",
             xaxis_title="規模 ML", yaxis_title="年發生率 N（M ≥ ML）",
             yaxis_type="log", hovermode="x")
fig

高規模端（$M\ge4.5$）四條曲線大致重合——大地震誰都漏不掉。分歧
全在低規模端：曲線偏離直線往下彎的位置就是各年代的 $M_c$，1970
年代落在 2.6 附近，1994 年後壓到 2.3，與 Tsai et al. (2012) 的
2.61 與 2.40 相當接近。要注意這份公開目錄的收錄下限本身就是
$M_L\,2.0$，所以 2012 年後真正的偵測能力（文獻報告可達 1.5）在
圖上看不出來。**估計方法只能看到資料讓它看到的東西。**

### 規模尺度換代：三條轉換式

上面四段用的是三種不同的規模，要併成一份可用的長期目錄，必須先做
**均一化**。台灣文獻裡最該記住的三條是：

$$\begin{aligned}
M_D &= 0.187 + 0.862\,M_L &&\text{(Wang et al. 1989)}\\
M_L &= -0.24 + 1.07\,M_w \pm 0.31 &&\text{(Chang et al. 2016)}\\
M_w &= 0.87\,M_L + 0.23 &&\text{(AutoBATS)}
\end{aligned}$$

後兩條不是彼此的反函數：反解第二條得 $M_w=0.935\,M_L+0.224$，斜率
與第三條的 0.87 差約 7%——兩條迴歸的資料集、規模範圍與迴歸方向都
不同。所以實務建議不是「選一條公式」，而是**建立優先序**：有
Global CMT 的 $M_w$ 就用它，其次 USGS，再其次由寬頻矩張量轉換，
最後才用經驗迴歸（Chang et al. 2016）。另一個轉換式救不回的問題是
**飽和**：$M_D$ 在 6.0 以上、$M_L$ 在 6.5 以上飽和，1978/07/23
（真值 $M_w\,7.2$）與 1986/11/14（真值 $M_w\,7.3$）在原目錄裡都被
低估，最大誤差可達一個規模單位；線性迴歸修不了飽和，只能換資料
來源。

### 尺度換代如何縮放 b 值

假設在 $M_D$ 尺度下 GR 律成立，$\log_{10}N(\ge M_D)=a'-b'M_D$，而
我們另有轉換式 $M_D=c_0+c_1M_L$（本例 $c_0=0.187$、$c_1=0.862$）。
同一批地震、同一個累積計數，只是換座標，直接代入：

$$\begin{aligned}
\log_{10}N &= a' - b'\left(c_0 + c_1 M_L\right)\\
           &= \left(a' - b'c_0\right) - \left(b'c_1\right)M_L
            \equiv a - b\,M_L .
\end{aligned}$$

比對係數即得縮放律 $a=a'-b'c_0$、$b=c_1b'$：**$b$ 值按轉換式的
斜率縮放、$a$ 值按截距平移**。

套進池上的例子（Chen et al. 2024）。Wang (1988) 用 $M_D$ 給出台東
縱谷南段的背景 $b'\approx1.1$；要與 2022 年池上序列以 $M_L$ 算出的
$b$ 值比較，必須先換算成 $b=0.862\times1.1=0.948\approx0.95$。有了
這個基準，池上的結果才讀得出意義：前震 $b=0.52$（MLE）／$0.62$
（最小二乘）、餘震 $b=0.84$／$0.87$，**兩者都低於背景**。若偷懶
直接拿 1.1 當基準，會得到「餘震低 0.26」；正確換算後是低 0.11
——結論的強度差了一倍以上。

最後補一句誠實話。同一批文獻記載台灣 1973–1987 的 $b$ 值約 0.83、
1994–2011 約 0.99（Tsai et al. 2012）。這個跳升方向與 0.862 的縮放
**相反**，所以**不能**用單一縮放因子解釋掉：那兩段期間同時發生了
規模尺度換代、$M_c$ 下降約 0.2、測站幾何全面改變三件事。多因素
同時改變時，能做的只有分段估計、分別報告，不要硬接成一條時間序列。

## 11.2 GR 律：連續與離散兩種寫法

Gutenberg–Richter 律的教科書寫法是累積形式 $N(\ge m)=10^{\,a-bm}$
（$m\ge m_0$），說的是「規模每降一級，次數增為 $10^b$ 倍」。但點
過程模型要的不是計數而是**機率密度**——條件強度必須對 $m$ 可積分。
橋樑只有兩行。令 $N_0=N(\ge m_0)=10^{a-bm_0}$，存活函數就是比例：

$$\begin{aligned}
S(m) &= \frac{10^{a-bm}}{10^{a-bm_0}} = 10^{-b(m-m_0)}\\
     &= \exp\left[-b\ln(10)(m-m_0)\right] = e^{-\beta(m-m_0)},
\end{aligned}$$

其中定義 $\beta\equiv b\ln 10\approx2.3026\,b$。密度是存活函數的
負導數：

$$
s(m) = -\frac{\mathrm{d}S}{\mathrm{d}m} = \beta\,e^{-\beta(m-m_0)},
\qquad m \ge m_0 .
$$ (eq:gr-density)

所以 **GR 律等價於「規模服從以 $m_0$ 為起點的指數分布」**，$\beta$
就是率參數。這個等價性是後面所有模型的共同基礎：ETAS 把
{eq}`eq:gr-density` 直接乘進條件強度（第 13 章），PSHA 用它的截斷
版本做規模積分（第 21 章）。全書一律用 $\beta$ 代表 $b\ln10$。

### a 值的三種定義

$b$ 值只有一個，$a$ 值至少有三種寫法，而文獻常常不說用哪一種
（$T$ 為目錄時間長度、$m_c$ 為所用門檻）：

| 定義 | 式子 | 意義 |
|---|---|---|
| 絕對 | $N(\ge m)=10^{a-bm}$ | 外推到 $m=0$ 的總數 |
| 相對 $m_c$ | $N(\ge m)=10^{a_c-b(m-m_c)}$ | $a_c=\log_{10}N(\ge m_c)$ |
| 單位時間率 | $\nu(\ge m)=10^{a_r-b(m-m_c)}$ | $a_r=a_c-\log_{10}T$ |

換算是純代數：$a_c=a-b\,m_c$、$a_r=a-b\,m_c-\log_{10}T$；換到另一
個參考規模也只是平移，$a_{m_{\rm ref}}=a_c-b(m_{\rm ref}-m_c)$。
三點提醒：絕對 $a$ 值外推到 $m=0$，本身沒有觀測意義，跨研究比較請
換到共同的 $m_{\rm ref}$；$a$ 值的換算**需要 $b$ 值**，所以 $b$ 的
誤差會傳染給 $a$，但多數文獻不報告 $a$ 值誤差（SeismoStats 的 a 值
估計器甚至刻意不提供 `std` 屬性）；$a$ 值也可按面積或體積正規化，
跨區比較才有意義，ETAS 的背景率 $\mu(x,y)$ 就是這樣定義的。

## 11.3 $M_c$ 是一個場，不是一個數

完整度規模 $M_c$ 是「目錄從這個規模以上大致收錄齊全」的門檻。它是
**目錄的性質**，不是地球的性質；依本書記號，$M_c$ 一律指估計出來的
完整度，模型設定的輸入門檻寫作 $m_0$（通常 $m_0\ge M_c$）。初學者
最常犯的錯是把它當常數塞進整份分析。它至少在三個維度上變動。

**時間**：上一節的圖已給答案，台灣目錄的 $M_c$ 從 1970 年代的約 2.6
降到 1994 年後的約 2.3–2.4。任何跨年代比較都必須先固定一個**共同
門檻**（取各年代 $M_c$ 的最大值），否則「地震變多」多半只是儀器
變好。

**空間**：本島測站密集，$M_c$ 可低到 1.5–2.0；外海涵蓋差，高達
2.5–3.2（Chan & Wu 2013）。差 1.7 個規模單位，依 GR 律換算約是
**50 倍的事件數差異**。對全目錄套單一 $M_c$，等於讓外海的不完整
資料污染整體統計；正確做法是估一張 $M_c$ 的空間場（0.2°×0.2° 網格
的 MAXC 地圖，或 Mignan et al. 2011 的 BMC 貝氏方法）再逐格套用。

**主震後短期（STAI）**：大地震後幾小時到幾天，小餘震的波形被主震
尾波淹沒而偵測不到——這是 short-term aftershock incompleteness，
效果等於 $M_c$ 短暫暴增再隨時間回落。常用經驗式（Helmstetter
et al. 2005）是 $M_c(t)=M_m-4.5-0.76\log_{10}t$，$M_m$ 為主震規模、
$t$ 為主震後天數。以 2024 年 0403 花蓮 $M_L\,7.2$ 為例：$t=0.01$ 天
（約 15 分鐘）時 $M_c\approx4.2$，$t=1$ 天時降到 2.7。台灣也有自己
的版本（Tsai et al. 2012），量級相當。**在這條曲線以下做任何統計，
都是在統計儀器而不是地震。**

### 三種估計法

**（一）最大曲率法（MAXC）**。取**非累積** FMD 的峰值規模再加保守
修正 $\delta_{\rm corr}=0.2$，即 $M_c=\arg\max_{m_i}n(m_i)+0.2$。
直覺是偵測率隨規模上升、GR 律隨規模下降，兩者相乘的峰值大致落在
完整度轉折處。「加 0.2」曾長期只是經驗慣例，直到 Tinti & Gasperini
(2024) 的模擬給出量化根據——用**規模本身**的估計式時確實需要
$M_c\ge M_{\rm maxc}+0.2$，而用**規模差**的估計式（11.7 節）時
$M_c\ge M_{\rm maxc}$ 就夠。MAXC 快、穩定、對小樣本友善，但對直方圖
格距敏感，FMD 雙峰時會被帶偏。實作上還有個常見混淆：**資料本身的
離散格距 $\Delta M$ 與畫直方圖的格寬 $\Delta M^{*}$ 是兩回事**。

**（二）$b$ 值穩定度判準**（Cao & Gao 2002；Woessner & Wiemer
2005）。利用「$M_c$ 取太低會低估 $b$、抬高門檻後 $b$ 上升並趨於
平台」的性質，取 $b$ 值開始穩定的那一點：

$$M_c = \min\left\{ m_i \;:\;
\left|\frac{1}{K}\sum_{k=1}^{K} b\!\left(m_i + k\Delta M^{*}\right)
- b(m_i)\right| < \sigma_{b(m_i)} \right\} .$$

意思是：往上再看 $L=K\Delta M^{*}$ 這麼寬的一段（預設 $L=0.5$），
$b$ 值的平均漂移小於當前估計的一個標準差就算穩定。它比 MAXC 更貼近
我們真正想要的東西，代價是**依賴 $\sigma_b$ 的估法**：樣本極大時
$\sigma_b$ 極小，判準變嚴苛，$M_c$ 被推高。

**（三）擬合優度／KS 距離**（Clauset et al. 2009；Mizrahi et al.
2021）。對每個候選門檻 $m_i$：(1) 取 $m\ge m_i$ 的子樣本，用 11.5
節的離散精確式估 $\hat b$；(2) 算觀測與理論累積分布的
Kolmogorov–Smirnov 距離
$D=\max_k\left|F_{\rm obs}(m_k)-F_{\rm mod}(m_k)\right|$；(3) 從擬合
出的分布**模擬** $S$ 份同樣大小的樣本，每份重估 $\hat b$、重算
$D_s$；(4) 模擬 p 值 $p=\#\{D_s\ge D\}/S$；(5) 取第一個滿足
$p\ge p_{\rm th}$（預設 0.1）的 $m_i$。

第 3 步的模擬是必要的：$\hat b$ 從同一批資料估出，KS 統計量的虛無
分布不再是標準 Kolmogorov 分布。這個「參數由資料估計、虛無分布由
模擬取得」的模式，11.8 節還會再出現一次。KS 路線的陷阱是**對樣本數
極度敏感**：目錄愈大，愈微小的偏離都被判為顯著，$M_c$ 被推得過高。
下面的實作刻意把每個候選門檻的樣本上限壓在 2,000 筆，就是在示範
這個限制。

In [ ]:
def b_exact(m, mc, dm=DM):
    """離散精確式（Tinti & Mulargia 1987）＋ Shi & Bolt (1982) 標準差。"""
    m = np.asarray(m, float)
    m = m[m >= mc - 1e-9]
    b = np.log1p(dm / (m.mean() - mc)) / (L10 * dm)
    return b, L10 * b ** 2 * np.sqrt(m.var(ddof=1) / len(m)), len(m)


def mc_bstability(m, grid, k_win=5):
    """b 值穩定度判準（Cao & Gao 2002；Woessner & Wiemer 2005）。"""
    bs, ss = np.array([b_exact(m, c)[:2] for c in grid]).T
    for i, c in enumerate(grid):
        if i + k_win < len(grid) and \
                abs(bs[i + 1:i + 1 + k_win].mean() - bs[i]) < ss[i]:
            return round(float(c), 2), bs, ss
    return float(grid[-1]), bs, ss


def mc_ks(m, grid, rng, nsim=400, nmax=2000, p_th=0.1):
    """Clauset/Mizrahi 式的模擬 p 值程序；nmax 為每個門檻的樣本上限。"""
    for c in grid:
        s = m[m >= c - 1e-9]
        if len(s) < 100:
            break
        if len(s) > nmax:
            s = rng.choice(s, nmax, replace=False)
        n, k = len(s), np.round((s - c) / DM).astype(int)
        r = 10 ** (-b_exact(s, c)[0] * DM)
        grid_k = np.arange(int(k.max()) + 1)
        f_mod = 1 - r ** (grid_k + 1)
        d_obs = np.abs(np.bincount(k, minlength=len(grid_k)).cumsum() / n
                       - f_mod).max()
        sim = np.sort(rng.geometric(1 - r, size=(nsim, n)) - 1, axis=1)
        f_sim = np.stack([np.searchsorted(row, grid_k, side="right")
                          for row in sim]) / n
        if float((np.abs(f_sim - f_mod).max(axis=1) >= d_obs).mean()) >= p_th:
            return round(float(c), 2)
    return float(grid[-1])


m94 = cat_long[(cat_long.time >= "1994") & (cat_long.time < "2012")].M.to_numpy()
grid_mc = np.round(np.arange(2.0, 4.51, DM), 2)
mc_a = maxc(m94)
mc_b, b_curve, s_curve = mc_bstability(m94, grid_mc)
mc_c = mc_ks(m94, grid_mc, np.random.default_rng(11))

counts, _ = np.histogram(m94, bins=BINS)
fig = go.Figure(go.Bar(x=CENTERS, y=counts, marker_color=ACCENT,
                       opacity=0.75, name="非累積 FMD"))
for x, name, color in [(mc_a, f"MAXC＋0.2 = {mc_a:.1f}", PALETTE[1]),
                       (mc_b, f"b 值穩定度 = {mc_b:.1f}", PALETTE[2]),
                       (mc_c, f"KS 模擬 p 值 = {mc_c:.1f}", PALETTE[3])]:
    fig.add_vline(x=x, line_color=color, line_dash="dash",
                  annotation_text=name, annotation_position="top")
apply_layout(fig,
             title=f"三種 Mc 估計法（台灣 1994–2011，N = {len(m94):,}）："
                   f"彼此差 {max(mc_a, mc_b, mc_c) - min(mc_a, mc_b, mc_c):.1f}"
                   f" 個規模單位",
             xaxis_title="規模 ML", yaxis_title="事件數",
             xaxis_range=[1.9, 5.5], yaxis_type="log", hovermode="x")
fig

三個方法給出三個答案。這不是誰壞掉了，而是**「完整」本來就沒有唯一
定義**：MAXC 問「哪裡的事件數最多」，穩定度判準問「從哪裡開始 $b$
值不再變」，KS 問「從哪裡開始指數分布不會被拒絕」。三個問題不同，
答案當然不同。穩定度判準給出最高的 $M_c$，理由前面說過：這段目錄有
十七萬筆事件，Shi & Bolt 標準差小到 0.004 左右，判準變得非常嚴苛。
**樣本大不代表判準會變好——有時只是變兇。**

實務建議是三個都跑、取中位數或最保守者，並且一定要做**敏感度
分析**。做這件事最直接的工具是 $b(M_c)$ 曲線：

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=np.r_[grid_mc, grid_mc[::-1]],
    y=np.r_[b_curve + s_curve, (b_curve - s_curve)[::-1]],
    fill="toself", fillcolor="rgba(42,120,214,0.20)", line=dict(width=0),
    hoverinfo="skip", name="±σ（Shi & Bolt）"))
fig.add_trace(go.Scatter(x=grid_mc, y=b_curve, mode="lines+markers",
                         name="b(Mc)", line=dict(color=ACCENT),
                         marker=dict(size=5)))
fig.add_vrect(x0=mc_b, x1=mc_b + 0.5, fillcolor=PALETTE[2], opacity=0.13,
              line_width=0, annotation_text="穩定度判準的 L = 0.5 視窗",
              annotation_position="top left")
for x, color in [(mc_a, PALETTE[1]), (mc_b, PALETTE[2]), (mc_c, PALETTE[3])]:
    fig.add_vline(x=x, line_color=color, line_dash="dash")
i_b = int(np.argmin(np.abs(grid_mc - mc_b)))
apply_layout(fig,
             title=f"b(Mc) 曲線與穩定度判準帶（1994–2011）：低門檻端由 "
                   f"{b_curve[0]:.2f} 爬升到平台 {b_curve[i_b]:.2f}",
             xaxis_title="截取門檻 Mc", yaxis_title="估計的 b 值",
             hovermode="x")
fig

曲線的形狀就是不完整性的指紋：門檻取太低時 $b$ 被壓低（缺漏的小
事件把分布尾巴削平了），隨門檻抬高而爬升，最後進入平台。三條虛線是
三種方法給的 $M_c$，淺綠色帶是穩定度判準實際比較的那段 $L=0.5$
視窗。誤差帶在右端明顯變寬，因為樣本數隨門檻指數衰減——這就是
「$M_c$ 取太高則樣本銳減、誤差爆炸」的視覺版本。要記得：平台的存在
只保證「$b$ 值不再隨門檻變」，不保證那個值是對的。下一節就要拆掉
這個保證。

## 11.4 Aki 最大概似的完整推導

有了 $M_c$ 就可以估 $b$ 值。歷史上第一個正確的做法來自 Aki (1965)：
不要對 FMD 做最小二乘迴歸（那會給低規模端的高計數過大權重，而且
殘差根本不是同質變異），改用最大概似。

假設 $M_c$ 以上的 $N$ 個規模獨立同分布、服從 {eq}`eq:gr-density` 的
指數密度（把 $m_0$ 換成 $M_c$），則
$L(\beta)=\prod_{i=1}^{N}\beta e^{-\beta(m_i-M_c)}$。取對數把連乘
變連加：

$$\begin{aligned}
\ln L(\beta) &= \sum_{i=1}^{N}\left[\ln\beta
               - \beta\left(m_i - M_c\right)\right]\\
             &= N\ln\beta - \beta N\left(\bar m - M_c\right),
\end{aligned}$$

其中 $\bar m$ 是樣本平均規模。注意只有 $\bar m$ 進得了式子——**樣本
平均是這個模型的充分統計量**，規模的排序、變異數、最大值對估
$\beta$ 毫無貢獻。對 $\beta$ 微分並令其為零：

$$\frac{\partial \ln L}{\partial \beta}
  = \frac{N}{\beta} - N\left(\bar m - M_c\right) = 0
  \quad\Longrightarrow\quad \hat\beta = \frac{1}{\bar m - M_c} .$$

二階導數 $\partial^2\ln L/\partial\beta^2=-N/\beta^2<0$，確認是極大
值。換回 $b$：

$$\hat b = \frac{\hat\beta}{\ln 10}
         = \frac{\log_{10}e}{\bar m - M_c}
         \approx \frac{0.4343}{\bar m - M_c} .$$

這是統計地震學裡被引用最多的一行式子。它漂亮、只需要一個平均數，
而且**兩個前提都是錯的**。

### 偏差之一：binning 讓 $b$ 被高估

真實目錄的規模記到小數點後一位，「$M_L=3.2$」的實際意思是「落在
$[3.15,3.25)$」。設格距 $\Delta M=2\delta$（台灣目錄 $\Delta M=0.1$、
$\delta=0.05$）。把連續指數分布按格中心分箱，落在第 $k$ 格
（$m=M_c+k\Delta M$）的機率是

$$P_k = \int_{M_c+k\Delta M-\delta}^{M_c+k\Delta M+\delta}
        \beta e^{-\beta(m-M_c+\delta)}\,\mathrm{d}m
      \propto e^{-\beta \Delta M\,k} = r^{k},
      \qquad r \equiv 10^{-b\Delta M} .$$

正規化後 $P_k=(1-r)r^k$——**分箱後的規模服從幾何分布**（離散指數
分布），而且是精確的、不是近似。由標準結果 $\mathbb{E}[k]=r/(1-r)$
（附錄 C）得 $\mathbb{E}[\bar m]-M_c=\Delta M\,r/(1-r)$。代進 Aki
公式，取 $b=1$、$\Delta M=0.1$：$r=10^{-0.1}=0.7943$，
$r/(1-r)=3.8615$，於是 $\bar m-M_c=0.38615$，而

$$\hat b_{\rm Aki} = \frac{0.4343}{0.38615} = 1.125 .$$

**12.5% 的系統性高估，完全不隨樣本數縮小。** Tinti & Gasperini
(2024) 的模擬在 $N=1000$、$b=1$、$\Delta M=0.1$ 時得到
$\bar b=1.126$，與解析值一致；他們的效能指標 $p=0.0006$，遠低於
0.05。這直接反駁了「binning 偏差可以忽略」這個流傳甚廣的說法。

### 偏差之二：不完整讓 $b$ 被低估

第二個前提是「目錄在 $M_c$ 以上完整」。$M_c$ 取太低時，靠近門檻的
小事件有一部分沒被記錄，$\bar m$ 被往上拉——分布的低端被削掉了
——於是 $\bar m-M_c$ 變大、$\hat b$ 變小。這個偏差沒有解析公式，但
方向是確定的，而且就是上一張 $b(M_c)$ 圖左半的斜坡。

### 兩個偏差可能互相抵銷

這是本節最需要記住的一句話。Binning 把 $b$ 往上推、不完整把 $b$
往下拉；在某些 $M_c$ 的選擇下兩者量級恰好相當，未修正的 Aki 公式會
給出「看起來很對」的答案。但這種正確是巧合——換一份目錄、換一個
格距、換一個門檻，比例就變了，而你**沒有任何辦法從結果本身判斷它
是不是又碰巧抵銷**。正確的做法是分別把兩個偏差消掉。**兩個錯不會
湊成一個對。**

In [ ]:
from scipy.stats import norm

rng = np.random.default_rng(42)
B_TRUE = 1.0
m_raw = rng.exponential(np.log10(np.e) / B_TRUE, 400_000) + 2.0
m_raw = np.round(m_raw / DM) * DM                     # 分箱到 0.1
obs = m_raw[rng.random(m_raw.size)
            < norm.cdf(m_raw, loc=3.0, scale=0.25)]   # 偵測率平滑上升

cutoffs = np.round(np.arange(2.6, 4.41, DM), 2)
b_aki = np.array([np.log10(np.e) / (obs[obs >= c - 1e-9].mean() - c)
                  for c in cutoffs])
b_exa = np.array([b_exact(obs, c)[0] for c in cutoffs])
d_pos = np.diff(obs)[np.diff(obs) > 0]
b_pos = {dc: np.log1p(DM / (d_pos[d_pos >= dc - 1e-9].mean() - dc))
             / (L10 * DM) for dc in (0.2, 0.6)}

fig = go.Figure()
fig.add_trace(go.Scatter(x=cutoffs, y=b_aki, mode="lines+markers",
                         name="Aki 連續式（未修 binning）",
                         line=dict(color=PALETTE[1]), marker=dict(size=5)))
fig.add_trace(go.Scatter(x=cutoffs, y=b_exa, mode="lines+markers",
                         name="離散精確式", line=dict(color=ACCENT),
                         marker=dict(size=5)))
for dc, color in [(0.2, PALETTE[4]), (0.6, PALETTE[2])]:
    fig.add_hline(y=b_pos[dc], line_color=color, line_dash="dot",
                  annotation_text=f"b-positive（trim {dc}）＝ {b_pos[dc]:.3f}")
fig.add_hline(y=B_TRUE, line_dash="dash", line_color=QUAKE_COLOR,
              annotation_text=f"真值 b = {B_TRUE:.1f}")
apply_layout(fig,
             title=f"合成不完整＋分箱目錄（N = {len(obs):,}）：Aki 高原 "
                   f"{b_aki.max():.3f}，精確式高原 {b_exa.max():.3f}",
             xaxis_title="截取門檻（假設的 Mc）", yaxis_title="估計的 b 值",
             hovermode="x")
fig

這張圖裝了本章前半的全部劇情。合成目錄真值 $b=1.0$，格距 0.1，
偵測率在規模 3.0 附近平滑劣化。**門檻取低時**（左半）兩條曲線都
嚴重低估——這是不完整造成的，跟用哪個公式無關。**門檻夠高後**
（右半）兩條曲線分家：精確式收斂到 1.00，Aki 式停在 1.125，正是
上面解析算出的 12.5%。

而在中間——大約門檻 3.1–3.2 附近——**Aki 曲線恰好穿過真值**。一位
剛好選了那個門檻又只用 Aki 公式的研究者會得到完全正確的答案，並且
完全不知道自己踩在兩個偏差的抵銷點上。這就是「假象，不是驗證」的
具體長相。另外兩條水平線是 11.7 節的 b-positive：不需要知道 $M_c$
在哪，就已經比左半段的傳統估計好很多。

## 11.5 離散精確式

既然分箱後的規模服從幾何分布（11.4 節已證），正確做法不是「修正
Aki 公式」，而是**對正確的模型重做一次最大概似**。幾何分布
$P_k=(1-r)r^k$ 的最大概似估計就是動差配對（$\bar k$ 是充分統計量）。
令 $u\equiv\bar m-M_c$，由 $\mathbb{E}[k]=r/(1-r)$ 與
$u=\Delta M\,\mathbb{E}[k]$ 得 $r=u/(u+\Delta M)$。再由
$r=10^{-b\Delta M}$ 反解：

$$b = -\frac{\log_{10} r}{\Delta M}
    = \frac{1}{\Delta M}\log_{10}\frac{u+\Delta M}{u}
    = \frac{1}{\Delta M \ln 10}\ln\left(1 + \frac{\Delta M}{u}\right).$$

用半格 $\delta=\Delta M/2$ 改寫就是文獻裡最常見的形式：

$$
b = \frac{1}{2\delta\ln 10}\,
    \ln\left(1 + \frac{2\delta}{\bar m - M_c}\right)
$$ (eq:b-exact)

這條式子有三個獨立來源（Guttorp & Hopkins 1986；Tinti & Mulargia
1987；Marzocchi & Sandri 2003），彼此代數等價。van der Elst (2021)
寫的是雙曲餘切反函數形式
$b=\frac{1}{\delta\ln10}\coth^{-1}\!\left(\frac{\bar m-M_c+\delta}{\delta}\right)$，
兩者相同（附錄 A 兩行證完）。**這是 binned 目錄的正確預設選項**，
也是本章之後所有計算採用的式子。當 $\Delta M\to0$ 時
$\ln(1+\Delta M/u)/\Delta M\to1/u$，它自動退化成 Aki 式——連續式
不是被推翻，而是被包含。

### Utsu 半格修正是精確式的二階截斷

精確式普及前最流行的修正來自 Utsu (1966)：把 $M_c$ 往下挪半格，
$b_{\rm Utsu}=\log_{10}e/(u+\delta)$。它為什麼有效？令
$\epsilon=2\delta/u$，由
$\ln(1+\epsilon)=\epsilon-\frac{\epsilon^2}{2}+\frac{\epsilon^3}{3}-\cdots$
得

$$\begin{aligned}
b_{\rm exact}
 &= \frac{1}{2\delta\ln10}\left(\epsilon - \frac{\epsilon^2}{2}
    + \frac{\epsilon^3}{3} - \cdots\right)
  = \frac{1}{\ln 10}\left(\frac{1}{u} - \frac{\delta}{u^2}
    + \frac{4\delta^2}{3u^3} - \cdots\right);\\
b_{\rm Utsu}
 &= \frac{1}{\ln 10}\cdot\frac{1}{u}\cdot\frac{1}{1 + \delta/u}
  = \frac{1}{\ln 10}\left(\frac{1}{u} - \frac{\delta}{u^2}
    + \frac{\delta^2}{u^3} - \cdots\right).
\end{aligned}$$

兩個級數**前兩項完全相同**，到第三項才分家（係數 $4/3$ 對 $1$）。
所以 Utsu 修正不是一個獨立的想法，它就是精確式展開到 $\delta$ 一階、
截斷掉 $\delta^2$ 的結果；截斷掉的項為正，因此 **Utsu 式系統性低估
$b$**。

何時失效？看展開參數 $\epsilon=\Delta M/(\bar m-M_c)$。台灣目錄
$\Delta M=0.1$、$b\approx1$ 時 $u\approx0.386$，$\epsilon\approx0.26$
——級數收斂快，Utsu 只低估約 0.4%，實務可接受。但由巨觀震度換算而來
的規模 $2\delta=0.5$ 時 $u\approx0.231$，$\epsilon\approx2.16>1$，
**級數根本不收斂**，「二階截斷」這個說法失去意義；Utsu 式給出
$0.4343/(0.231+0.25)=0.902$，低估 10%，模擬顯示它連效能檢定的門檻
都過不了。結論很簡單：**格距大於 0.1 一定要用精確式**；格距是 0.1
時用精確式也不多花力氣（一行 `np.log1p`）。沒有理由不用。

## 11.6 不確定度的三代

估計值只是一半，另一半是誤差。這裡的三個公式代表三個世代，每一代
都在修前一代的一個問題。

### 第一代：Aki 的 $\sigma_b = b/\sqrt N$

由第 10 章的 Fisher 資訊直接得到。對 11.4 節的對數概似再微分一次，
$\partial^2\ln L/\partial\beta^2=-N/\beta^2$，故

$$I(\beta) = -\mathbb{E}\!\left[\frac{\partial^2 \ln L}
             {\partial\beta^2}\right] = \frac{N}{\beta^2},
\qquad \operatorname{Var}(\hat\beta)\approx\frac{1}{I(\beta)}
     = \frac{\beta^2}{N}.$$

於是 $\sigma_\beta=\beta/\sqrt N$；除以 $\ln10$ 換回 $b$，兩邊同時
縮放，得 $\sigma_b=b/\sqrt N$。問題不在代數，在於它**假設模型完全
正確**：只用到 $N$ 與 $b$，完全沒看資料的實際離散程度。

### 第二代：Shi & Bolt (1982)

改用資料自己的變異數。對 $b=\log_{10}e/u$ 用差量法，先算導數
$\mathrm{d}b/\mathrm{d}u=-\log_{10}e/u^2=-b^2\ln10$；樣本平均的
標準誤是 $\sigma_u=s/\sqrt N$，其中
$s^2=\frac{1}{N-1}\sum_i(m_i-\bar m)^2$。相乘：

$$\sigma_b = \left|\frac{\mathrm{d}b}{\mathrm{d}u}\right|\sigma_u
           = \ln(10)\,b^2\,
             \sqrt{\frac{\sum_{i=1}^{N}\left(m_i-\bar m\right)^2}
                         {N\,(N-1)}} .$$

這正是 Shi & Bolt 的原式，而且有個漂亮的自洽性質：規模真的服從指數
分布時母體標準差恰等於平均值 $u$，代入後**完全退回 Aki 式**
（附錄 D）。所以 Shi & Bolt 不是另一個模型，而是「不要假設模型對，
去看資料」的版本；兩者的差距本身就是一個診斷量——差很多，代表你的
規模分布不像指數。

### 第三代：離散情形的非對稱 1σ 區間

前兩代都給一個 $\pm\sigma$，暗示誤差對稱。它不是。Tinti & Gasperini
(2024) 給出離散情形的區間端點。令 $c=10^{2\delta\tilde b}$
（$\tilde b$ 為 {eq}`eq:b-exact` 的估計值）：

$$\begin{aligned}
b_1 &= \frac{1}{2\delta\ln 10}
       \ln\left[\frac{c + c/\sqrt N}{1 + c/\sqrt N}\right],
&b_2 &= \frac{1}{2\delta\ln 10}
       \ln\left[\frac{c - c/\sqrt N}{1 - c/\sqrt N}\right],
\end{aligned}$$

而慣用的單一標準差取兩臂平均，
$\sigma=(\sigma_1+\sigma_2)/2=(b_2-b_1)/2$，其中
$\sigma_1=\tilde b-b_1$、$\sigma_2=b_2-\tilde b$。適用條件是
$N\gtrsim30$–40（讓 $\bar m$ 近似常態）與 $N>c^2$。

**為什麼 $\sigma_2>\sigma_1$ 是系統性的？** 因為 $b$ 是 $\bar m$ 的
**凸遞減函數**：由 {eq}`eq:b-exact` 可直接驗證 $f'(u)<0$ 而
$f''(u)>0$（附錄 E）。把 $\bar m$ 上一個對稱的誤差區間映射過去時，
$u$ 偏小的那一側（對應 $b$ 偏大）被拉得比較開，另一側被壓得比較扁
——**$b$ 的區間右臂一定比左臂長**。同一個凸性還告訴我們（Jensen
不等式）$\mathbb{E}[f(\bar m)]>f(\mathbb{E}[\bar m])$，即 $\hat b$
在小樣本時**平均而言偏高**。

In [ ]:
cat24 = pd.read_csv(CACHE_DIR / "catalog_2024spring.csv", parse_dates=["time"])
main = cat24.loc[cat24.ML.idxmax()]
win = cat24[(cat24.time > main.time + pd.Timedelta(days=1))
            & (cat24.time <= main.time + pd.Timedelta(days=30))]
pool = np.round(win.ML.to_numpy() / DM) * DM
pool = pool[pool >= 3.5 - 1e-9]

rng_bs = np.random.default_rng(2024)
samp = rng_bs.choice(pool, 60, replace=False)
b_hat, sig_sb, n_sub = b_exact(samp, 3.5)
boot = np.array([b_exact(rng_bs.choice(samp, n_sub, replace=True), 3.5)[0]
                 for _ in range(2000)])

c_val = 10 ** (DM * b_hat)
sN = c_val / np.sqrt(n_sub)
b1 = np.log((c_val + sN) / (1 + sN)) / (L10 * DM)
b2 = np.log((c_val - sN) / (1 - sN)) / (L10 * DM)

fig = go.Figure(go.Histogram(x=boot, xbins=dict(size=0.02),
                             marker_color=ACCENT, opacity=0.8,
                             name="bootstrap 分布"))
fig.add_vline(x=b_hat, line_color=QUAKE_COLOR,
              annotation_text=f"點估計 {b_hat:.3f}")
for x, dash, color, name in [(b_hat - sig_sb, "dot", PALETTE[1],
                              "Shi & Bolt ±σ"),
                             (b_hat + sig_sb, "dot", PALETTE[1], ""),
                             (b1, "dash", PALETTE[2], f"b1 = {b1:.3f}"),
                             (b2, "dash", PALETTE[2], f"b2 = {b2:.3f}")]:
    fig.add_vline(x=x, line_dash=dash, line_color=color, annotation_text=name)
apply_layout(fig,
             title=f"bootstrap 的 b 值分布（0403 花蓮餘震子樣本 N = {n_sub}）："
                   f"σ1 = {b_hat - b1:.3f} 對 σ2 = {b2 - b_hat:.3f}",
             xaxis_title="估計的 b 值", yaxis_title="次數", hovermode="x")
fig

直方圖明顯右偏：上半臂比下半臂長。點狀線是 Shi & Bolt 的對稱
$\pm\sigma$，虛線是離散非對稱區間 $[b_1,b_2]$。對稱誤差棒在左側
**過寬**、右側**過窄**，系統性地低估了「$b$ 值其實可能更高」這個
可能性。這對「前震 $b$ 值比較低」這類宣稱有直接影響：若兩個 $b$ 值
的差異只比一個對稱 $\sigma$ 大一點點，換成正確的非對稱區間後很可能
就不顯著了。

## 11.7 差分法家族

到目前為止所有估計式都需要一個可信的 $M_c$。但 $M_c$ 在主震後會
短暫暴增（STAI），而餘震序列的早期正好是資料最密、最誘人拿來算
$b$ 值的一段。有沒有辦法**繞開 $M_c$**？van der Elst (2021) 的答案
是：不要用規模本身，改用**規模差**。全部根據是下面這個結果。

### 兩個獨立指數變數之差服從 Laplace 分布

設 $X,Y$ 獨立、都服從率為 $\beta$ 的指數分布（起點無所謂，相減時
共同位移會消掉），令 $D=X-Y$。對 $d\ge0$ 做卷積：

$$\begin{aligned}
f_D(d) &= \int_{0}^{\infty} f_X(y+d)\,f_Y(y)\,\mathrm{d}y
        = \int_{0}^{\infty} \beta e^{-\beta(y+d)}\,
          \beta e^{-\beta y}\,\mathrm{d}y\\
       &= \beta^{2} e^{-\beta d}\int_{0}^{\infty}
          e^{-2\beta y}\,\mathrm{d}y
        = \beta^{2} e^{-\beta d}\cdot\frac{1}{2\beta}
        = \frac{\beta}{2}\,e^{-\beta d} .
\end{aligned}$$

$D$ 的分布對 $d\to-d$ 對稱（$X,Y$ 同分布），所以
$f_D(d)=\frac{\beta}{2}e^{-\beta|d|}$，即**尺度為 $1/\beta$ 的
Laplace 分布**。關鍵在於位移消掉了：**規模差的分布完全不依賴
$M_c$**。只要在任一時刻兩個被記錄到的事件都來自同一條指數尾巴，
它們的差就帶著正確的 $\beta$，不管當時的完整度門檻在哪裡。

### 三個直接推論

**絕對差**：$P(|D|>x)=e^{-\beta x}$，所以 $|D|$ 本身是率為 $\beta$
的指數變數，$\mathbb{E}|D|=1/\beta$，連續情形下
$b=\log_{10}e/\overline{|\Delta M|}$——分母不必減掉任何門檻。
**單號差**：條件於 $D>0$，$P(D>d\mid D>0)=e^{-\beta d}$，又是指數
分布，取正差與取負差在理論上完全等價。**trimming**：指數分布無
記憶，$P(|D|>d_c+x \mid |D|>d_c)=e^{-\beta x}$，所以丟掉
$|\Delta M|<\Delta M'_c$ 的差之後剩下的仍是指數分布、只是起點移到
$\Delta M'_c$，估計式因而與規模的精確式**同形**：

$$b = \frac{1}{2\delta\ln 10}
      \ln\left[\frac{\overline{|\Delta M|} - \Delta M'_c + 2\delta}
                    {\overline{|\Delta M|} - \Delta M'_c}\right] .$$

離散情形下未做 trimming 的絕對差需要另一條式子，因為零差在離散
Laplace 分布裡有額外權重、不屬於幾何尾巴（附錄 F）：

$$b = \frac{1}{2\delta\ln 10}\,
      \mathrm{csch}^{-1}\!\left(\frac{\overline{|\Delta M|}}{2\delta}\right)
    = \frac{1}{2\delta\ln 10}
      \ln\left[\frac{2\delta + \sqrt{4\delta^2
        + \overline{|\Delta M|}^{\,2}}}{\overline{|\Delta M|}}\right] .$$

一旦執行最低限度的 trimming（$\Delta M'_c=2\delta$，即只丟掉零差），
分布退化成移位幾何分布，回到上面那條同形式子。

### 三個成員

**b-positive**（van der Elst 2021）取**相鄰**事件的規模差
$\Delta M_i=m_{i+1}-m_i$，只保留 $\Delta M_i\ge\delta m_c$ 的正差再套
最大概似。物理假設是：**任一時刻的完整度門檻，至多是上一個被記錄到
的規模再加一點餘裕**——這對 STAI 相當合理，因為主震後偵測不到小
事件正是因為前一個大事件的波形還在。它解決 STAI，**解決不了**觀測網
偵測能力長期不足這種一般性不完整（Lippiello & Petrillo 2024）；補救
方式是先截到 $M_c$ 之上再取差，或加大 $\delta m_c$。

**b-more-positive**（Lippiello & Petrillo 2024）改成對每個 $m_i$ 往後
找**第一個**滿足 $m_j\ge m_i+\delta m_c$ 的事件取差，可用的差比
b-positive 多。代價是差之間的相關性更複雜，合成測試發現實際標準差
大於理論值，所以現代實作（SeismoStats）對它一律改用 bootstrap 算誤差。

**a-positive**（van der Elst & Page 2023）把同樣的想法搬到 $a$ 值：
$a$ 值本質是單位時間的事件數，不完整期間會少算，解法是只計入「確實
被使用到的 inter-event 時間」佔總觀測時間的比例，
$a^{+}=\log_{10}n^{+}-\log_{10}\left(\sum_{i=1}^{n^{+}}\Delta t_i/T\right)$。
a-more-positive 進一步依 GR 律縮放時間差
$\tau_i=\Delta t_i\,10^{-b(m_i+\delta m_c)}$，並把「後面再也沒有更大
事件」的開放區間 $T_j=(T-t_j)\,10^{-b(m_j+\delta m_c)}$ 一併算進去以免
系統性偏差。

### 取差的方式：(A) 相鄰 vs (B) 配對

這是實作上最容易踩的坑。兩種取法：

$$\begin{aligned}
\text{(A)}\quad \Delta M_i &= m_{i+1} - m_i, && i = 1,\dots,N-1;\\
\text{(B)}\quad \Delta M_i &= m_{2i} - m_{2i-1}, && i = 1,\dots,N/2 .
\end{aligned}$$

(A) 用光所有相鄰對、資料量最大，但每個 $m_i$ 同時出現在兩個差裡，
**引入相關性**；(B) 每個事件只用一次、保證獨立，但差的數量減半。
Tinti & Gasperini (2024) 把代價量化了：用**絕對差**時，(A) 的理論
標準差比實際散布小了約 22–23%，等於有效樣本數只剩
$N_e\approx0.67N$——所以絕對差**必須用 (B)**，否則誤差棒會系統性
太小。但用**單號差**時相關性的影響消失（理論 σ 與實際散布之比約為
1），因此單號差**應該用 (A)**，否則資料會被砍到四分之一。

### trimming 才是關鍵，不是取正號

van der Elst 原本主張「非得取正差不可」。Tinti & Gasperini (2024) 在
合成不完整目錄、合成餘震序列、以及 2016 年義大利 Norcia $M_w\,6.6$
真實序列三個層次上檢驗，都找不到正差優於絕對差或負差的證據（Norcia
三種取法給出 $1.04\pm0.05$、$1.03\pm0.05$、$1.01\pm0.05$，彼此無法
區分）。結論是：**真正有效的是 trimming 門檻**，而且付出的樣本數
代價遠比提高 $M_c$ 溫和。

In [ ]:
def b_trim(d, dc):
    """trimmed 差分估計式。"""
    d = np.asarray(d, float)
    d = d[d >= dc - 1e-9]
    return np.log1p(DM / (d.mean() - dc)) / (L10 * DM)


rng_af = np.random.default_rng(7)
N_SYN, C_OM, T_OM, M_MAIN, M0 = 200_000, 0.01, 30.0, 5.6, 2.0
u_om = rng_af.random(N_SYN)
t_af = np.sort(C_OM * ((1 + T_OM / C_OM) ** u_om - 1))     # Omori（p = 1）
k_af = rng_af.geometric(1 - 10 ** (-B_TRUE * DM), size=N_SYN) - 1
m_af = np.round(M0 + k_af * DM, 2)
mc_t = np.maximum(M_MAIN - 4.5 - 0.76 * np.log10(np.maximum(t_af, 1e-4)), M0)
m_obs = m_af[rng_af.random(N_SYN) < norm.cdf(m_af, loc=mc_t, scale=0.5)]

b_plain = b_exact(m_obs, M0)[0]
dd = np.diff(m_obs)                                    # 取法 (A)：相鄰
half = len(m_obs) // 2 * 2
d_pair = np.abs(m_obs[1:half:2] - m_obs[0:half:2])     # 取法 (B)：配對
trims = np.round(np.arange(0.1, 1.01, 0.1), 2)

fig = go.Figure()
for name, arr, color in [("正差（取法 A）", dd[dd > 0], PALETTE[0]),
                         ("負差（取法 A）", -dd[dd < 0], PALETTE[1]),
                         ("絕對差（取法 B）", d_pair, PALETTE[2])]:
    fig.add_trace(go.Scatter(x=trims, y=[b_trim(arr, dc) for dc in trims],
                             mode="lines+markers", name=name,
                             line=dict(color=color), marker=dict(size=6)))
fig.add_hline(y=B_TRUE, line_dash="dash", line_color=QUAKE_COLOR,
              annotation_text=f"真值 b = {B_TRUE:.1f}")
fig.add_hline(y=b_plain, line_dash="dot", line_color="#888888",
              annotation_text=f"直接用規模的精確式 = {b_plain:.3f}")
apply_layout(fig,
             title=f"trimming 門檻掃描（合成餘震序列，觀測到 {len(m_obs):,} "
                   f"個事件）：三種取差方式一起收斂",
             xaxis_title="trimming 門檻 ΔM'c", yaxis_title="估計的 b 值",
             hovermode="x")
fig

合成序列模仿 Tinti & Gasperini 的檢驗情境：$M_m=5.6$ 的主震、Omori
$p=1$、$c=0.01$ 天、30 天窗，完整度依 11.3 節的 STAI 經驗式隨時間
回落，另加寬度 0.5 的平滑偵測機率（真實的偵測邊界不是一刀切）。

灰色點線是直接用規模的精確式：公式本身是對的，但仍低估約 20%——
問題不在 binning，在**不完整**。三條彩色曲線是三種取差方式，幾乎
重疊，並且**隨 trimming 門檻加大一起單調收斂到真值**。正差沒有比較
好，負差沒有比較差。這張圖還藏著一個反直覺的教訓：Tinti & Gasperini
報告，在完整度隨時間變化的序列上 $N\gtrsim4000$ 的大樣本估計反而
**比小樣本更差**——大樣本裡混進更多時間不完整的早期資料，偏差不會
被平均掉，只會被固化。**系統偏差不隨 $\sqrt N$ 縮小**；加大 trimming
之後這個現象才消失。

## 11.8 前提本身可以被檢定

前面七節做了一件事：把 $b$ 值估得愈來愈準。但整套推導從第一行起就
架在一個假設上——**規模服從指數分布**。這個假設可以檢查，而且有兩個
現成的工具。

**Lilliefors 檢定**（Lilliefors 1969；Herrmann & Marzocchi 2021）是
KS 檢定的變種，專門處理「參數是從同一批資料估出來的」這個情況。標準
KS 檢定假設理論分布完全指定；一旦用樣本估了 $\hat\beta$ 再去比對，
KS 統計量會系統性偏小（模型被拉去貼合資料了），用標準臨界值會**過度
接受**虛無假設。Lilliefors 的修正是虛無分布改由模擬取得——從擬合出的
指數分布重複抽樣、每次重估參數、重算 KS 距離。這個結構跟 11.3 節的
KS 式 $M_c$ 估計完全一樣，只是目的不同：那裡是**選門檻**，這裡是
**檢定分布形狀**。

**b-significance 方法**（Mirwald et al. 2024）處理另一個問題：宣稱
$b$ 值有時空變化之前，得先確認變化幅度超過估計本身的不確定度。這聽
起來理所當然，但文獻裡大量「$b$ 值異常」根本沒過這一關——尤其是滑動
窗畫出的 $b(t)$ 曲線，相鄰窗共用大部分事件，波動被人為放大。

兩個檢定合起來，等於替「$b$ 值在變」設了兩道舉證關卡：規模分布
**真的是**指數嗎（Lilliefors）？變化幅度**真的超過**不確定度嗎
（b-significance）？順序不能顛倒——如果規模分布根本不是指數，那麼
「$b$ 值」這個量本身就沒有定義，談它的變化是無意義的。Herrmann &
Marzocchi (2020) 分析南加州與義大利中部的高解析目錄後指出，這個前提
未必總是成立。

```{admonition} 報告 b 值時的最低要求
:class: tip
任何 $b$ 值的報告，至少要同時附上：所用的 $M_c$ 與**它是怎麼決定
的**、規模尺度與格距 $\Delta M$、樣本數 $N$、估計式（Aki／Utsu／
精確式／差分法）、誤差的算法（Aki／Shi & Bolt／非對稱區間），以及
是否除叢與用了哪一種。少任何一項，這個 $b$ 值都無法被別人重現，
也就無法被比較。
```

## 11.9 參數與典型值

以下是**文獻報告值**，供對照與健全性檢查；本章圖上的數字一律由程式
以 f-string 帶入，兩者不一定相同（資料窗、$M_c$、深度範圍都可能不同）。

| 年代 | 規模尺度 | $M_c$ | $b$ |
|---|---|---|---|
| 1973–1987.5 | $M_{D(A)}$ | 約 2.61 | 約 0.83 |
| 1987.6–1991.2 | $M_{D(D)}$ | 約 3.0 | — |
| 1994–2011 | $M_L$ | 約 2.40 | 約 0.99 |
| 2012– | $M_L$ | 1.5–2.0（陸上） | — |

出處：Tsai et al. (2012)、Wang et al. (2015)。全台長期平均
$b\approx1.0$–$1.1$，隨時段、尺度與分區而異。

| $M_c$ 的其他維度 | 值 | 出處 |
|---|---|---|
| 陸上（測站密集） | 1.5–2.0 | Chan & Wu (2013) |
| 外海（測站涵蓋差） | 2.5–3.2 | 同上 |
| 主震後初期（STAI） | 可達 4.0 以上 | Tsai et al. (2012) |
| 作業型 ETAS 設定 | $M_L\,3.0$ | Hsieh et al. (2025) |

| 適用期／來源 | 規模轉換式 |
|---|---|
| 1900–1972（$M_H$） | $M_H = -1.239 + 1.207\,M_w$ |
| 1973–1987.6 | $M_{D(A)} = 1.316 + 0.720\,M_w \pm 0.43$ |
| 1987.6–1991.2 | $M_{D(D)} = 0.758 + 0.720\,M_w \pm 0.37$ |
| 1991.3– | $M_L = -0.24 + 1.07\,M_w \pm 0.31$ |
| 跨期 $b$ 值換算 | $M_D = 0.187 + 0.862\,M_L$ |

出處：Chang et al. (2016)、Wang et al. (1989)。AutoBATS 的
$M_w = 0.87\,M_L + 0.23$ 見 11.1 節。

| 個案序列 | $b$（前震／餘震／背景） | 出處 |
|---|---|---|
| 2022 池上 $M_L\,6.8$ | 0.52 / 0.84 / 0.95 | Chen et al. (2024) |
| 1983 太平山 $M_D\,5.7$ | 0.99 / 1.20 / 1.13 | Wang et al. (2015) |
| 2008 桃源 $M_L\,5.2$ | 1.25 / 0.80 / 0.81 | 同上 |

注意這三列方向並不一致：池上與太平山的餘震 $b$ 高於前震，桃源反過來。
**這種不一致本身就是重點**——個案的 $b$ 值差異在被當成前兆之前，
必須先通過 11.8 節的兩道關卡。

## 11.10 常見誤解與陷阱

這一節把近年文獻推翻或大幅削弱的「常識」集中起來。它們的共同特徵是：
都曾寫進教科書，都在最近十年被系統性的模擬或對照實驗打了折扣。

- **「主震的 $b$ 值天生比較低。」** Mizrahi et al. (2021) 用五類除叢法
  處理加州目錄，發現除叢後的主震 $b$ 值比全目錄低了最多 30%；拿
  「$b$ 值設計上完全均一」的合成目錄重跑也一樣下降。原因是**選擇
  效應**：多數演算法把叢集中最大的事件定義為主震，而小地震本來就
  不容易成為一群地震裡最大的那個。最乾淨的對照是他們的
  ETAS-Background 實驗——改用「非被觸發者」定義主震，$b$ 值與全目錄
  就沒有顯著差異。除叢的完整討論留給第 12 章。
- **「binning 偏差很小，可以忽略。」** 11.4 節解析算出 12.5% 的高估，
  模擬也證實（$\bar b=1.126$，效能指標 $p=0.0006$），而且不隨樣本數
  縮小。
- **「Utsu 修正夠用了。」** 它只是精確式的二階截斷（11.5 節有完整
  證明）。格距 0.1 時誤差約 0.4%，格距 0.5 時展開參數大於 1、級數不
  收斂，低估達 10%。
- **「b-positive 必須用正差。」** 合成資料與 Norcia 真實序列都顯示
  正差、負差、絕對差表現相當；有效的是 trimming 門檻。
- **「樣本愈大估計愈準。」** 只有在偏差為零時才成立。完整度隨時間
  變化時大樣本反而更差——系統偏差不隨 $\sqrt N$ 縮小。
- **「$b$ 值的時空變化就是應力變化的訊號。」** 必須先過 Lilliefors 與
  b-significance 兩關。$M_c$ 的選擇、規模型別混用、採石場爆破都會製造
  假訊號。Marzocchi et al. (2020) 有一篇論文的標題就叫〈How to be
  fooled searching for significant variations of the b-value〉——標題
  本身就值得抄在實驗室牆上。
- **「軟體會幫我檢查資料。」** 不會。SeismoStats 的作者自己在論文裡
  寫明了已知地雷：$M_c$ 的時空變化（套件所有 $M_c$ 方法都假設 $M_c$
  不隨時間變）、人為事件、目錄裡混雜多種規模型別。工具負責算得對，
  不負責替你判斷輸入是否合理。

如果只能記住一句話：**$b$ 值對 $M_c$ 的選擇極度敏感。取太低則 $b$ 被
低估，取太高則樣本銳減、誤差爆炸。任何沒有附上 $M_c$ 決定方式與敏感度
分析的 $b$ 值報告，都不能當成結論。**

## 11.11 研究前沿與未解問題

**$b$ 值能不能當前震的紅綠燈？** 這是目前最受關注也最有爭議的應用。
Gulia & Wiemer (2019) 提出**前震／餘震交通號誌（foreshock traffic
light system, FTLS）**：序列開始後持續估計 $b$ 值並與該區背景值比較。
若相對背景**下降**，代表區域應力仍高、目前的事件可能只是更大事件的
前震（紅燈）；若**上升**，代表應力已釋放、比較可能是一般餘震序列
（綠燈）。物理基礎不是憑空來的：實驗室岩石破裂實驗長期觀察到 **$b$
值隨差應力升高而下降**（Scholz 1968 以降的一系列工作），這給了「低
$b$ 值 = 高應力」一個可檢驗的機制。台灣也有方向一致的觀察：2022 池上
序列的前震 $b$（0.52）明顯低於餘震（0.84）與換算後的背景（0.95）；
Wetzler et al. (2023) 在全球八個區域得到「前震 $b$ 值低 0.1–0.2」的
一致結果。

但這條路線上的所有障礙，都是本章前八節列出的那些。序列早期正是 STAI
最嚴重的時候；滑動窗會人為放大波動；背景值需要跨尺度換算（11.1 節）；
「下降」要多大才算數，取決於 11.6 節那個非對稱區間怎麼算。更根本的
是：**這些成功案例幾乎全是回溯分析**，而回溯分析的自由度遠大於即時
判定。要讓 FTLS 從有趣的觀察變成可用的工具，需要預先註冊、前瞻性的
檢驗——也就是第 17、18 章要建立的那套文化。

**舉證責任的移轉。** 更廣泛地說，統計地震學這十年最大的方法論轉向是
舉證責任的移轉。過去畫出一條 $b(t)$ 曲線、指出一個低谷，就算發現了
一個現象；現在的標準是：先證明分布是指數的、再證明變化超過不確定度、
再證明結果對 $M_c$ 與時間窗的選擇穩健。這個轉向還沒完成，而且有個
實際困難：符合標準的分析往往得出「不顯著」的結論，而不顯著的結果
不容易發表。這是統計地震學版本的發表偏差問題，目前沒有解法。

**工具生態。** 這一整章的內容正在被打包成標準軟體。**SeismoStats**
（Mirwald et al. 2025）是 ETH 蘇黎世團隊為取代已停止維護的 MATLAB 版
ZMAP 而寫的 Python 套件，把 $M_c$ 三法、四種 $b$ 值估計器（離散精確
式、Utsu、b-positive、b-more-positive）、Shi & Bolt 標準差、Lilliefors
與 b-significance 檢定都實作成統一介面。它在生態系裡的位置是：ObsPy
管波形、SeisComP 管即時、SeisBench 管機器學習、pyCSEP 管預報檢定、
OpenQuake 管危害度，而 SeismoStats 補的是「目錄的統計分析」這一塊。
即使最新的套件也還有已知缺口：它對所有方法（b-more-positive 除外）
統一使用 Shi & Bolt 標準差，而那條式子是在連續規模假設下推導的；
11.6 節第三代的離散非對稱區間目前還沒被實作。**文獻已經解決、軟體
還沒跟上**——這種落差在任何活著的領域裡都是常態，也是讀者現在就能
貢獻的地方。

## 11.12 附錄：本章推導細節

**A. $\coth^{-1}$ 形式與 {eq}`eq:b-exact` 的等價性。** 由
$\coth^{-1}x=\frac12\ln\frac{x+1}{x-1}$。令 $u=\bar m-M_c$、
$x=(u+\delta)/\delta$，則
$\frac{x+1}{x-1}=\frac{(u+\delta)/\delta+1}{(u+\delta)/\delta-1}
=\frac{u+2\delta}{u}$，故

$$\frac{1}{\delta\ln10}\coth^{-1}x
 = \frac{1}{2\delta\ln10}\ln\left(1+\frac{2\delta}{u}\right).$$

**B. Utsu 截斷誤差的係數。** 把 11.5 節的兩個級數相減：

$$b_{\rm exact} - b_{\rm Utsu}
 = \frac{1}{\ln 10}\left(\frac{4}{3} - 1\right)\frac{\delta^2}{u^3}
   + O\!\left(\frac{\delta^3}{u^4}\right)
 = \frac{\delta^2}{3\,u^3\ln 10} + \cdots$$

恆為正，所以 Utsu 式低估 $b$。但這個估計只在 $\epsilon=2\delta/u<1$
時有意義；$\epsilon\ge1$ 時級數發散，只能直接比較數值。

**C. 幾何分布的期望值與 $\hat\beta$ 的小樣本偏差。**

$$\mathbb{E}[k] = \sum_{k=0}^{\infty} k\,(1-r)r^{k}
                = (1-r)\,r\cdot\frac{\mathrm{d}}{\mathrm{d}r}
                  \left(\frac{1}{1-r}\right) = \frac{r}{1-r}.$$

另外，$\hat\beta=1/(\bar m-M_c)$ 的分母服從 Gamma$(N,\beta)$，其倒數
的期望值給出 $\mathbb{E}[\hat\beta]=\frac{N}{N-1}\beta$——最大概似
估計量**系統性偏高**，要無偏就乘上 $(N-1)/N$。

**D. Shi & Bolt 退回 Aki。** 指數分布的標準差等於平均值，故大樣本時
$s\to u=1/(b\ln10)$，代入：

$$\sigma_b = \ln(10)\,b^2\,\frac{s}{\sqrt N}
          \to \ln(10)\,b^2\cdot\frac{1}{b\ln 10}\cdot\frac{1}{\sqrt N}
           = \frac{b}{\sqrt N}.$$

**E. $b$ 是 $\bar m$ 的凸遞減函數。** 令
$f(u)=\frac{1}{2\delta\ln10}\ln\left(1+\frac{2\delta}{u}\right)$：

$$\begin{aligned}
f'(u) &= \frac{-1}{\ln 10}\cdot\frac{1}{u\,(u+2\delta)} < 0,
&f''(u) &= \frac{1}{\ln 10}\cdot
           \frac{2u + 2\delta}{u^2\,(u+2\delta)^2} > 0 .
\end{aligned}$$

因此對稱誤差映射到 $b$ 上必然右臂較長，且由 Jensen 不等式估計量平均
偏高。連續情形（$\delta\to0$，$f(u)=\log_{10}e/u$）結論相同。

**F. 離散 Laplace 分布。** 設 $k_1,k_2$ 獨立且服從 $P(k)=(1-r)r^k$，
令 $j=k_1-k_2$。對 $j\ge0$：

$$P(j) = \sum_{k=0}^{\infty}(1-r)r^{\,k+j}\,(1-r)r^{k}
       = \frac{(1-r)^2}{1-r^2}\,r^{\,j}
       = \frac{1-r}{1+r}\,r^{\,|j|}.$$

$j=0$ 的機率 $(1-r)/(1+r)$ 並不落在幾何尾巴的自然延伸上——這正是未
trimming 的絕對差需要 $\mathrm{csch}^{-1}$ 形式的原因。丟掉零差後，
$P(|j|=n \mid |j|\ge1)=(1-r)r^{\,n-1}$ 又回到移位幾何分布。

**G. 相關性與有效樣本數。** 取法 (A) 的相鄰差共用一個規模，
$\operatorname{Cov}(\Delta M_i,\Delta M_{i+1})=-1/\beta^2$，而
$\operatorname{Var}(\Delta M)=2/\beta^2$，故相關係數為 $-1/2$。取絕對
值後負相關不會消失，於是 $\overline{|\Delta M|}$ 的變異數不等於獨立
情形的 $\operatorname{Var}(|\Delta M|)/N$；模擬把整體效應量化為理論 σ
偏小 22–23%，等價於 $N_e\approx0.67N$。

**H. $b$ 值尺度換算的一般式。** 若 $M_2=\alpha M_1+\beta_0$
（$\alpha>0$），以 $M_2$ 表示的 GR 律為 $\log_{10}N=a_2-b_2M_2$，則
$\log_{10}N=(a_2-b_2\beta_0)-(\alpha b_2)M_1$，故 $b_1=\alpha\,b_2$、
$a_1=a_2-b_2\beta_0$。三條台灣轉換式的縮放因子分別是 0.862、1.07、
0.87；注意後兩者不互為倒數（$1/1.07=0.935\ne0.87$），因為兩條迴歸的
資料集與方向不同。跨尺度比較 $b$ 值時，**要說明用了哪一條、往哪個
方向換算**。

---

回頭看，這一章只做了一件事：把「$b$ 值」這三個字拆開，看看裡面裝了
什麼。裝的是一個目錄的行政史（規模欄位換過三次定義）、一個估計不到
的門檻（$M_c$ 是場不是數）、一個被四捨五入破壞的連續假設、一對方向
相反的偏差、一組不對稱的誤差棒，以及一個可以被檢定、也應該被檢定的
分布假設。

這聽起來像在拆台。但反過來想：這些坑之所以能被一條一條指出來，正是
因為這個領域花了六十年把它們一個一個踩過。Aki 在 1965 年寫下的那條
式子沒有被推翻，它被**包含**了——離散精確式在 $\Delta M\to0$ 時退化
成它，Shi & Bolt 在模型正確時退化成它。每一代修正都不是否定前一代，
而是把前一代的適用範圍標清楚。這是知識健康累積的樣子。

規模軸講完了，還剩時間軸。同一批目錄、同一批陷阱，換到時間維度上會
長出另一組經驗律：餘震怎麼衰減（Omori–Utsu 的 $c$ 與 $p$）、最大餘震
有多大（Båth）、以及那個最麻煩的操作——把目錄拆成「背景」與「叢集」
的除叢，它會像 11.10 節預告的那樣，回頭污染我們剛剛辛苦估好的 $b$
值。這些是 {doc}`第 12 章 <12_clustering_laws>`的內容。